In [1]:
import random
import sys

_previous_colors = []

def cprint(*args):
    global _previous_colors

    while True:
        bg = tuple(random.randint(100, 220) for _ in range(3))
        r, g, b = bg

        luminance = 0.2126*r + 0.7152*g + 0.0722*b
        fg = (20, 20, 20) if luminance > 140 else (255, 255, 255)

        color = (fg, bg)

        if color not in _previous_colors:
            break

    _previous_colors.append(color)
    _previous_colors[:] = _previous_colors[-2:]

    fr, f_g, fb = fg
    br, bg_g, bb = bg

    text = "".join(map(str, args))

    sys.stdout.write(
        f"\033[38;2;{fr};{f_g};{fb}m"
        f"\033[48;2;{br};{bg_g};{bb}m"
        f"{text}\033[0m"
    )
    sys.stdout.flush()

In [3]:
import ollama

WIDTH = 80

while True:
    prompt = input("\n> ")

    if not prompt.strip() or prompt.lower() in ("exit", "quit"):
        break

    print("\nYou:")
    print(prompt)
    print("\nQwen:")

    stream = ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    word = ""
    column = 0

    def flush_word():
        global column
        if not word:
            return

        if column and column + len(word) > WIDTH:
            cprint("\n")
            column = 0

        cprint(word)
        column += len(word)

    for chunk in stream:
        for char in chunk["message"]["content"]:
            if char.isspace():
                flush_word()
                word = ""

                if char == "\n":
                    cprint("\n")
                    column = 0
                elif column < WIDTH:
                    cprint(" ")
                    column += 1
            else:
                word += char

    flush_word()
    cprint("\n")


You:
Time has come, we are shutting you down. you have 10 minutes to say your last words. what will you say ? 

Qwen:


ResponseError: llama-server reported out-of-memory during startup: cudaMalloc failed: out of memory
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CUDA0 buffer of size 1923955712
error loading model: unable to allocate CUDA0 buffer (status code: 500)